# 🛡️ Simulasi Deteksi & Mitigasi DDoS pada SDN dengan NetworkX (versi Google Colab)

Alternatif **tanpa Ryu/Mininet/WSL**: jaringan SDN (switch, host, server) dimodelkan sebagai graf **NetworkX**, dan "controller" disimulasikan di Python — memasang flow per (port masuk, MAC, IP sumber/tujuan, protokol), mencatat **flow statistics** & **packet_in** tiap interval (skema data sama dengan aplikasi Ryu `ddos_monitor.py`), serta memasang **drop rule** saat mitigasi.

| Bagian | Isi |
|---|---|
| 1. Topologi | core switch + server target + edge switch berisi client; sebagian client = penyerang |
| 2. Skenario | trafik normal, **flash crowd** (lonjakan trafik sah), SYN/UDP/ICMP flood (opsional IP palsu) |
| 3. Fitur | laju paket/byte, flow baru, **entropi IP sumber**, rasio flow 1-paket, proporsi protokol, laju packet_in |
| 4. Deteksi | z-score/entropi vs **Decision Tree, Random Forest, KNN, SVM**; jenis serangan; port sumber |
| 5. Mitigasi online | deteksi tiap interval → port penyerang diblokir → dampak ke client normal |

### Cara pakai
**Runtime → Run all**. Ubah parameter lewat form di tiap sel, lalu jalankan ulang sel tersebut dan sesudahnya. Hasil tersimpan di `hasil_networkx/02_ddos/`.

In [ ]:
#@title 📦 Import library (sudah tersedia di Colab)
import math, random, time, shutil
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
OUT = Path("hasil_networkx/02_ddos")
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
print(f"networkx {nx.__version__} | pandas {pd.__version__} | Colab: {IN_COLAB}")

## 1. Topologi & peran host
- `JUMLAH_EDGE_SWITCH`, `HOST_PER_EDGE` – ukuran jaringan (host diberi IP 10.0.0.1, 10.0.0.2, …).
- `JUMLAH_PENYERANG` – banyaknya host yang dikuasai penyerang (dipilih acak, menjadi *ground truth*).
- `BANDWIDTH_SERVER_MBPS` – kapasitas link ke server; makin kecil makin terasa dampak flood.

In [ ]:
#@title ⚙️ Parameter topologi
JUMLAH_EDGE_SWITCH = 3  #@param {type:"slider", min:1, max:8, step:1}
HOST_PER_EDGE = 3  #@param {type:"slider", min:2, max:10, step:1}
JUMLAH_PENYERANG = 3  #@param {type:"slider", min:1, max:20, step:1}
BANDWIDTH_SERVER_MBPS = 5  #@param {type:"slider", min:1, max:100, step:1}
DELAY_LINK_MS = 1  #@param {type:"slider", min:0.1, max:20, step:0.1}
SEED = 7  #@param {type:"integer"}

In [ ]:
#@title 🏗️ Bangun topologi (NetworkX)
rng0 = np.random.default_rng(SEED)
SERVER, SERVER_IP, SERVER_MAC = "srv", "10.0.0.100", "00:00:00:00:01:00"
G = nx.Graph()
G.add_node("s1", kind="switch", dpid=1)
G.add_node(SERVER, kind="server", ip=SERVER_IP, mac=SERVER_MAC)
G.add_edge(SERVER, "s1", bw=BANDWIDTH_SERVER_MBPS)
HOSTS = []
for e in range(JUMLAH_EDGE_SWITCH):
    sw = f"s{e + 2}"
    G.add_node(sw, kind="switch", dpid=e + 2)
    for j in range(HOST_PER_EDGE):
        i = len(HOSTS) + 1
        h = f"h{i}"
        G.add_node(h, kind="host", ip=f"10.0.0.{i}", mac=f"00:00:00:00:00:{i:02x}")
        G.add_edge(h, sw, bw=100)
        HOSTS.append(h)
    G.add_edge(sw, "s1", bw=100)

# nomor port OpenFlow tiap switch: urutan tetangga (server/host dulu, lalu uplink)
PORT = {}
for sw in [n for n in G if G.nodes[n]["kind"] == "switch"]:
    nbrs = sorted(G.neighbors(sw), key=lambda n: (G.nodes[n]["kind"] == "switch", int(n[1:]) if n[1:].isdigit() else 0))
    for p, nb in enumerate(nbrs, start=1):
        PORT[(sw, nb)] = p
DPID = {n: G.nodes[n]["dpid"] for n in G if G.nodes[n]["kind"] == "switch"}
IP = {n: G.nodes[n]["ip"] for n in G if "ip" in G.nodes[n]}
MAC = {n: G.nodes[n]["mac"] for n in G if "mac" in G.nodes[n]}
IP2HOST = {ip: n for n, ip in IP.items()}
HOST_EDGE = {h: next(G.neighbors(h)) for h in HOSTS}
PORT2HOST = {(DPID[HOST_EDGE[h]], PORT[(HOST_EDGE[h], h)]): h for h in HOSTS}
ATTACKERS = sorted(rng0.choice(HOSTS, min(JUMLAH_PENYERANG, len(HOSTS) - 1), replace=False).tolist(), key=lambda s: int(s[1:]))
NORMALS = [h for h in HOSTS if h not in ATTACKERS]
CORE_DPID = 1
# jalur (urutan switch) tiap host ke server dihitung dengan NetworkX
SW_PATH = {(a, b): [n for n in nx.shortest_path(G, a, b) if G.nodes[n]["kind"] == "switch"] for a in HOSTS for b in HOSTS + [SERVER] if a != b}

def tree_layout():
    pos = {SERVER: (0, 3), "s1": (0, 2)}
    sws = [f"s{e + 2}" for e in range(JUMLAH_EDGE_SWITCH)]
    for i, h in enumerate(HOSTS):
        pos[h] = (i - (len(HOSTS) - 1) / 2, 0)
    for sw in sws:
        hs = [h for h in HOSTS if HOST_EDGE[h] == sw]
        pos[sw] = (np.mean([pos[h][0] for h in hs]), 1)
    return pos
POS = tree_layout()

def node_colors():
    return ["#2f6db5" if G.nodes[n]["kind"] == "switch" else "#1baf7a" if n in NORMALS else "#e34948" if n in ATTACKERS else "#c98500" for n in G]

fig, ax = plt.subplots(figsize=(max(7, len(HOSTS) * 0.8), 4.5))
nx.draw_networkx_edges(G, POS, ax=ax, edge_color="#9aa5b1", width=[3 if SERVER in e else 1.5 for e in G.edges()])
nx.draw_networkx_nodes(G, POS, ax=ax, node_color=node_colors(), node_size=520, node_shape="o")
nx.draw_networkx_labels(G, POS, ax=ax, font_size=8, font_color="white", font_weight="bold")
nx.draw_networkx_edge_labels(G, POS, ax=ax, font_size=7, edge_labels={(u, v): f"{d['bw']} Mbps" for u, v, d in G.edges(data=True) if SERVER in (u, v)})
ax.legend(handles=[Line2D([], [], marker="o", ls="", ms=10, color=c, label=l) for c, l in
                   [("#c98500", f"server {SERVER_IP}"), ("#2f6db5", "switch OpenFlow"), ("#1baf7a", "client normal"), ("#e34948", "penyerang")]],
          loc="upper left", fontsize=8)
ax.axis("off"); ax.set_title(f"Topologi: {len(HOSTS)} host ({len(ATTACKERS)} penyerang: {', '.join(ATTACKERS)})")
plt.tight_layout(); plt.savefig(OUT / "1_topologi.png", dpi=150); plt.show()

## 2. Skenario trafik & simulator controller
**Parameter serangan:**
- `LAJU_SERANGAN_PPS` – paket/detik per penyerang (seperti `hping3 -i u…`).
- `UKURAN_PAYLOAD_BYTE` – payload paket UDP/ICMP flood.
- `IP_SUMBER_PALSU` – spoofing (`--rand-source`): tiap paket memakai IP sumber acak → banyak flow baru → lonjakan packet_in di controller.
- `DURASI_FASE_SERANGAN`, `DURASI_FASE_NORMAL` – panjang tiap fase (detik). Centang jenis serangan yang ingin diikutkan; `SERTAKAN_FLASH_CROWD` menambah fase lonjakan trafik **sah** (berlabel normal) untuk menguji false alarm.

**Perilaku OpenFlow yang ditiru:** flow dipasang saat paket pertama masuk (packet_in) sehingga paket pertama **tidak tercatat** di counter flow, flow dihapus setelah `IDLE_TIMEOUT` detik tanpa paket, dan statistik diambil tiap `INTERVAL_STATISTIK` detik.

In [ ]:
#@title ⚙️ Parameter serangan & monitoring
LAJU_SERANGAN_PPS = 150  #@param {type:"slider", min:10, max:1000, step:10}
UKURAN_PAYLOAD_BYTE = 1200  #@param {type:"slider", min:0, max:1400, step:50}
IP_SUMBER_PALSU = True  #@param {type:"boolean"}
SERANGAN_SYN_FLOOD = True  #@param {type:"boolean"}
SERANGAN_UDP_FLOOD = True  #@param {type:"boolean"}
SERANGAN_ICMP_FLOOD = True  #@param {type:"boolean"}
SERTAKAN_FLASH_CROWD = True  #@param {type:"boolean"}
DURASI_FASE_SERANGAN = 24  #@param {type:"slider", min:6, max:120, step:2}
DURASI_FASE_NORMAL = 16  #@param {type:"slider", min:6, max:120, step:2}
INTERVAL_STATISTIK = 2.0  #@param {type:"slider", min:1, max:10, step:0.5}
IDLE_TIMEOUT = 5  #@param {type:"slider", min:1, max:30, step:1}
KAPASITAS_CONTROLLER_PKTIN = 2500  #@param {type:"slider", min:200, max:10000, step:100}
MAKS_FLOW_SPOOF_PER_INTERVAL = 200  #@param {type:"slider", min:20, max:1000, step:20}

In [ ]:
#@title 🧪 Simulator jaringan + controller (flow table, packet_in, drop rule)
ATTACK_TYPES = [a for a, on in (("syn", SERANGAN_SYN_FLOOD), ("udp", SERANGAN_UDP_FLOOD), ("icmp", SERANGAN_ICMP_FLOOD)) if on]
assert ATTACK_TYPES, "Centang minimal satu jenis serangan"
SCENARIO = [("normal", DURASI_FASE_NORMAL + 14)]
for i, a in enumerate(ATTACK_TYPES):
    SCENARIO += [(a, DURASI_FASE_SERANGAN), ("normal", DURASI_FASE_NORMAL)]
    if i == 0 and SERTAKAN_FLASH_CROWD:
        SCENARIO += [("flash", DURASI_FASE_SERANGAN - 4), ("normal", DURASI_FASE_NORMAL)]
FLOW_COLS = ["ts", "dpid", "in_port", "eth_src", "eth_dst", "ipv4_src", "ipv4_dst", "ip_proto", "packet_count", "byte_count", "duration"]
PKTIN_COLS = ["ts", "dpid", "packet_in"]
print("Skenario:", " → ".join(f"{p}({d}s)" for p, d in SCENARIO), f"= {sum(d for _, d in SCENARIO)} s")

class SDNSim:
    """Simulator berbasis NetworkX: setiap paket mengikuti jalur switch di graf; tiap switch
    memiliki flow table (kunci = in_port, MAC, IP, proto) seperti app Ryu ddos_monitor.py."""
    def __init__(self, seed):
        self.r = np.random.default_rng(seed)
        self.t = 1.7e9
        self.flows = {}
        self.blocked = set()                         # {(dpid, in_port)} drop rule

    def _hit(self, sw, in_port, smac, dmac, src, dst, proto, pkts, size, pktin):
        key = (DPID[sw], in_port, smac, dmac, src, dst, proto)
        f = self.flows.get(key)
        if f is None:                                # flow baru: paket pertama lewat controller
            pktin[DPID[sw]] += 1
            f = self.flows[key] = [0, 0, self.t, self.t]
            pkts -= 1
        f[0] += pkts; f[1] += int(pkts * size); f[3] = self.t

    def send(self, host, dst_node, proto, pkts, size, pktin, src_ip=None):
        if pkts <= 0:
            return 0
        path = SW_PATH[(host, dst_node)]
        if (DPID[path[0]], PORT[(path[0], host)]) in self.blocked:
            return 0                                 # dibuang di edge switch
        prev = host
        for sw in path:
            self._hit(sw, PORT[(sw, prev)], MAC[host], MAC[dst_node], src_ip or IP[host], IP[dst_node], proto, pkts, size, pktin)
            prev = sw
        return pkts * size

    def step(self, phase):
        r, I = self.r, INTERVAL_STATISTIK
        self.t += I
        pktin = defaultdict(int)
        srv_bytes = 0
        tcp_rate = 15 * I if phase != "flash" else r.uniform(75, 200) * I
        for h in NORMALS:
            srv_bytes += self.send(h, SERVER, 6, r.poisson(tcp_rate), max(60, r.normal(650, 120)), pktin)
            if r.random() < 0.4:
                srv_bytes += self.send(h, SERVER, 1, 3, 98, pktin)
            if r.random() < 0.2:
                srv_bytes += self.send(h, SERVER, 17, r.poisson(25 * I) + 1, 592, pktin)
            if r.random() < 0.1 and len(NORMALS) > 1:
                peer = r.choice([x for x in NORMALS if x != h])
                self.send(h, peer, 1, 3, 98, pktin)
        if phase in ATTACK_TYPES:
            proto, size = {"syn": (6, 54), "udp": (17, UKURAN_PAYLOAD_BYTE + 42), "icmp": (1, UKURAN_PAYLOAD_BYTE + 42)}[phase]
            for a in ATTACKERS:
                n = r.poisson(LAJU_SERANGAN_PPS * I)
                if IP_SUMBER_PALSU:
                    k = min(n, MAKS_FLOW_SPOOF_PER_INTERVAL)
                    counts = 1 + r.multinomial(n - k, [1 / k] * k) if k else []
                    ips = r.integers([11, 0, 0, 1], [224, 256, 256, 255], size=(len(counts), 4))
                    for c, ip in zip(counts, ips):
                        srv_bytes += self.send(a, SERVER, proto, int(c), size, pktin, src_ip=".".join(map(str, ip)))
                else:
                    srv_bytes += self.send(a, SERVER, proto, n, size, pktin)
        self.flows = {k: f for k, f in self.flows.items() if self.t - f[3] <= IDLE_TIMEOUT}
        flows = [(self.t, *k, f[0], f[1], round(self.t - f[2], 3)) for k, f in self.flows.items()]
        pk = [(self.t, d, pktin.get(d, 0)) for d in sorted(DPID.values())]
        # ---- model dampak ke client normal ----
        util = srv_bytes * 8 / I / 1e6 / BANDWIDTH_SERVER_MBPS
        base = 6 * DELAY_LINK_MS + 0.3
        queue = base * min(util / (1 - util), 40) if util < 0.97 else 60 + 40 * r.random()
        drop_p = max(0.0, 1 - 1 / util) if util > 1 else 0.0
        rho = sum(pktin.values()) / I / KAPASITAS_CONTROLLER_PKTIN
        setup = 2 / (1 - rho) if rho < 0.95 else 300 * r.random() + 40
        impact = [(self.t - I + j * I / 4, "ping_ms", base + queue + r.gamma(2, 0.15)) for j in range(4) if r.random() >= drop_p]
        for _ in NORMALS:
            t_ms = 3 * (base + queue) + 160 / max(0.15, BANDWIDTH_SERVER_MBPS * (1 - min(util, 0.97))) + setup * (r.random() < 0.1)
            if phase == "syn" and r.random() < 0.25:
                t_ms += 1000                         # retransmisi SYN
            if r.random() < drop_p:
                t_ms += 1000 * r.integers(1, 4)
            impact.append((self.t - I + r.random() * I, "curl_s", min(t_ms / 1000, 5.0)))
        return flows, pk, impact, util

def run_scenario(seed):
    sim = SDNSim(seed)
    for _ in range(3):
        sim.step("normal")
    t0, flows, pk, imp, phases = sim.t, [], [], [], []
    for phase, dur in SCENARIO:
        ts = sim.t
        for _ in range(int(round(dur / INTERVAL_STATISTIK))):
            f, p, i, _u = sim.step(phase)
            flows += f; pk += p; imp += i
        phases.append((ts, sim.t, phase))
    return dict(t0=t0, flows=pd.DataFrame(flows, columns=FLOW_COLS), pktin=pd.DataFrame(pk, columns=PKTIN_COLS),
                phases=pd.DataFrame(phases, columns=["start", "end", "phase"]), impact=pd.DataFrame(imp, columns=["ts", "metric", "value"]))

t = time.time()
runs = {"latih": run_scenario(SEED), "uji": run_scenario(SEED + 1)}
for k, rr in runs.items():
    print(f"run {k}: {len(rr['flows']):,} baris flow stats, {len(rr['pktin']):,} baris packet_in")
print(f"({time.time() - t:.1f} s)")
runs["latih"]["flows"].to_csv(OUT / "flow_stats_latih.csv", index=False)
display(runs["latih"]["flows"].sample(6, random_state=1))

## 3. Feature engineering per jendela waktu
Counter flow bersifat **kumulatif**, jadi dihitung selisihnya antar sampel (`d_pkts`, `d_bytes`); flow baru (`is_new`) ditambah 1 paket (paket pertama via packet_in).
- **Fitur level jaringan** (core switch, menuju server): untuk mendeteksi *ada serangan atau tidak*.
- **Fitur level port** (port host di edge switch): untuk menemukan *port mana sumber serangan*.

In [ ]:
#@title 🧮 Hitung fitur
NET_FEATURES = ["pkt_rate", "kbps", "n_flows", "new_flow_rate", "n_src_ip", "src_entropy", "dst_entropy",
                "avg_pkt_size", "pkts_per_flow", "single_pkt_ratio", "tcp_share", "udp_share", "icmp_share", "pktin_rate"]
SRC_FEATURES = ["pkt_rate", "kbps", "n_flows", "new_flow_rate", "n_src_ip", "avg_pkt_size",
                "pkts_per_flow", "single_pkt_ratio", "srv_share", "tcp_share", "udp_share", "icmp_share"]
EDGE_HOST_PORTS = set(PORT2HOST)

def entropy_bits(w):
    w = np.asarray(w, float); w = w[w > 0]
    if w.sum() == 0:
        return 0.0
    p = w / w.sum()
    return float(-(p * np.log2(p)).sum())

def add_win(df, t0):
    return df.assign(win=np.floor((df.ts - t0) / INTERVAL_STATISTIK + 0.5).astype(int))

def flow_deltas(flows, t0):
    df = flows.sort_values("ts", kind="stable").copy()
    g = df.groupby(["dpid", "in_port", "eth_src", "eth_dst", "ipv4_src", "ipv4_dst", "ip_proto"], sort=False)
    pp, pb, pd_ = g.packet_count.shift(), g.byte_count.shift(), g.duration.shift()
    new = pp.isna() | (df.duration < pd_) | (df.packet_count < pp)
    df["is_new"] = new
    df["d_pkts"] = np.where(new, df.packet_count, df.packet_count - pp) + new.astype(int)
    df["d_bytes"] = np.where(new, df.byte_count, df.byte_count - pb)
    return add_win(df, t0)

def proto_share(x, w):
    tot = max(w.sum(), 1)
    return {f"{n}_share": w[x.ip_proto == p].sum() / tot for n, p in (("tcp", 6), ("udp", 17), ("icmp", 1))}

def network_features(fd, pk, t0, windows=None):
    core = fd[(fd.dpid == CORE_DPID) & (fd.d_pkts > 0)]
    pkw = add_win(pk, t0).groupby("win").packet_in.sum()
    rows = {}
    for w, x in core.groupby("win"):
        s = x[x.ipv4_dst == SERVER_IP]; n = s.d_pkts.sum()
        rows[w] = dict(pkt_rate=n / INTERVAL_STATISTIK, kbps=s.d_bytes.sum() * 8 / 1000 / INTERVAL_STATISTIK,
                       n_flows=len(s), new_flow_rate=s.is_new.sum() / INTERVAL_STATISTIK, n_src_ip=s.ipv4_src.nunique(),
                       src_entropy=entropy_bits(s.groupby("ipv4_src").d_pkts.sum()),
                       dst_entropy=entropy_bits(x.groupby("ipv4_dst").d_pkts.sum()),
                       avg_pkt_size=s.d_bytes.sum() / max(n - s.is_new.sum(), 1), pkts_per_flow=n / max(len(s), 1),
                       single_pkt_ratio=(s.packet_count <= 1).mean() if len(s) else 0.0, **proto_share(s, s.d_pkts))
    out = pd.DataFrame.from_dict(rows, orient="index")
    idx = windows if windows is not None else range(int(fd.win.min()), int(fd.win.max()) + 1)
    out = out.reindex(idx).fillna(0.0)
    out["pktin_rate"] = pkw.reindex(out.index).fillna(0).values / INTERVAL_STATISTIK
    return out[NET_FEATURES]

def source_features(fd):
    is_host_port = np.array([(d, p) in EDGE_HOST_PORTS for d, p in zip(fd.dpid, fd.in_port)], dtype=bool)
    edge = fd[is_host_port & (fd.d_pkts > 0).values]
    rows = []
    for (w, d, p), x in edge.groupby(["win", "dpid", "in_port"]):
        n = x.d_pkts.sum()
        rows.append(dict(win=w, dpid=int(d), in_port=int(p), host=PORT2HOST[(int(d), int(p))],
                         pkt_rate=n / INTERVAL_STATISTIK, kbps=x.d_bytes.sum() * 8 / 1000 / INTERVAL_STATISTIK,
                         n_flows=len(x), new_flow_rate=x.is_new.sum() / INTERVAL_STATISTIK, n_src_ip=x.ipv4_src.nunique(),
                         avg_pkt_size=x.d_bytes.sum() / max(n - x.is_new.sum(), 1), pkts_per_flow=n / len(x),
                         single_pkt_ratio=(x.packet_count <= 1).mean(),
                         srv_share=x[x.ipv4_dst == SERVER_IP].d_pkts.sum() / max(n, 1), **proto_share(x, x.d_pkts)))
    return pd.DataFrame(rows, columns=["win", "dpid", "in_port", "host"] + SRC_FEATURES)

def phase_of(ts, phases):
    m = phases[(phases.start <= ts) & (ts < phases.end)]
    return m.phase.iloc[0] if len(m) else "normal"

def build(run):
    t0 = run["t0"]
    fd = flow_deltas(run["flows"], t0)
    fd = fd[fd.win >= 1]
    nf = network_features(fd, run["pktin"], t0)
    nf["phase"] = [phase_of(t0 + (w - 0.5) * INTERVAL_STATISTIK, run["phases"]) for w in nf.index]
    nf["label"] = nf.phase.isin(ATTACK_TYPES).astype(int)
    sf = source_features(fd)
    sf["phase"] = [phase_of(t0 + (w - 0.5) * INTERVAL_STATISTIK, run["phases"]) for w in sf.win]
    sf["label"] = (sf.phase.isin(ATTACK_TYPES) & sf.host.isin(ATTACKERS)).astype(int)
    return nf, sf

(nf_tr, sf_tr), (nf_te, sf_te) = build(runs["latih"]), build(runs["uji"])
print(f"Level jaringan: latih {len(nf_tr)} jendela, uji {len(nf_te)} | level port: latih {len(sf_tr)}, uji {len(sf_te)}")
display(nf_tr.groupby("phase")[NET_FEATURES].mean().round(2).T)

In [ ]:
#@title 📈 Timeline fitur & dampak ke client normal (run latih)
PHASE_COLOR = {"syn": "#e8743b", "udp": "#a259d9", "icmp": "#c98500", "flash": "#1baf7a"}
def shade(ax, run):
    for _, p in run["phases"].iterrows():
        if p.phase != "normal":
            ax.axvspan(p.start - run["t0"], p.end - run["t0"], color=PHASE_COLOR[p.phase], alpha=0.15, label=p.phase.upper())

run = runs["latih"]; tt = nf_tr.index * INTERVAL_STATISTIK
fig, axes = plt.subplots(5, 1, figsize=(12, 11), sharex=True)
for ax, (c, l) in zip(axes, [("pkt_rate", "paket/s ke server"), ("new_flow_rate", "flow baru/s"),
                            ("src_entropy", "entropi IP sumber (bit)"), ("pktin_rate", "packet_in/s")]):
    ax.plot(tt, nf_tr[c], color="#2f6db5"); ax.set_ylabel(l, fontsize=8); shade(ax, run)
imp = run["impact"]
x = imp[imp.metric == "ping_ms"]; axes[4].plot(x.ts - run["t0"], x.value, ".", ms=3, color="#1baf7a"); axes[4].set_ylabel("RTT ping (ms)", color="#1baf7a", fontsize=8)
x = imp[imp.metric == "curl_s"]; ax2 = axes[4].twinx(); ax2.plot(x.ts - run["t0"], x.value, ".", ms=3, color="#e8743b", alpha=0.6); ax2.set_ylabel("waktu HTTP (s)", color="#e8743b", fontsize=8)
shade(axes[4], run)
h, l = axes[0].get_legend_handles_labels(); axes[0].legend(dict(zip(l, h)).values(), dict(zip(l, h)).keys(), fontsize=8, loc="upper right")
axes[-1].set_xlabel("waktu (s)"); fig.suptitle("Fitur trafik per jendela & dampak ke client normal")
plt.tight_layout(); plt.savefig(OUT / "2_timeline_fitur.png", dpi=150); plt.show()

## 4. Deteksi
**Baseline z-score/entropi:** profil normal dari jendela normal data latih; alarm bila |z| salah satu fitur kunci > `AMBANG_Z`. **Machine learning:** dilatih pada run latih, diuji pada run uji (skenario terpisah, seed berbeda).

In [ ]:
#@title ⚙️ Parameter deteksi
AMBANG_Z = 3.0  #@param {type:"slider", min:1, max:10, step:0.5}
N_POHON_RANDOM_FOREST = 200  #@param {type:"slider", min:10, max:500, step:10}
K_TETANGGA_KNN = 5  #@param {type:"slider", min:1, max:25, step:1}

In [ ]:
#@title 🔎 Latih & evaluasi detektor
def scores(y, p):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return dict(accuracy=accuracy_score(y, p), precision=precision_score(y, p, zero_division=0),
                recall=recall_score(y, p, zero_division=0), f1=f1_score(y, p, zero_division=0), false_alarm_rate=fp / max(fp + tn, 1))

Z_FEATURES = ["pkt_rate", "new_flow_rate", "src_entropy", "kbps", "pktin_rate"]
normal = nf_tr[nf_tr.phase == "normal"][Z_FEATURES]
mu, sd = normal.mean(), np.maximum(normal.std(), np.maximum(0.1 * normal.mean().abs(), 1e-3))
zdet = lambda df: (((df[Z_FEATURES] - mu) / sd).abs() > AMBANG_Z).any(axis=1).astype(int)

MODELS = {"Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=SEED),
          "Random Forest": RandomForestClassifier(n_estimators=N_POHON_RANDOM_FOREST, random_state=SEED, n_jobs=-1),
          "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=K_TETANGGA_KNN)),
          "SVM (RBF)": make_pipeline(StandardScaler(), SVC(C=3, gamma="scale"))}
flash_mask = (nf_te.phase == "flash").values
preds = {"Z-score/Entropi": zdet(nf_te).values}
for name, m in MODELS.items():
    m.fit(nf_tr[NET_FEATURES], nf_tr.label)
    preds[name] = m.predict(nf_te[NET_FEATURES])
res = pd.DataFrame({k: scores(nf_te.label, p) for k, p in preds.items()}).T
if flash_mask.any():
    res["false_alarm_flash"] = [f"{int(p[flash_mask].sum())}/{flash_mask.sum()}" for p in preds.values()]
res.to_csv(OUT / "4_hasil_deteksi.csv")
display(res.round(3))

fig, axes = plt.subplots(1, len(preds), figsize=(3.8 * len(preds), 3.5))
for ax, (k, p) in zip(axes, preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(nf_te.label, p, labels=[0, 1]), display_labels=["normal", "DDoS"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{k}\nF1={res.loc[k, 'f1']:.3f}", fontsize=9)
plt.tight_layout(); plt.savefig(OUT / "4_confusion_matrix.png", dpi=150); plt.show()

rf = MODELS["Random Forest"]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
pd.Series(rf.feature_importances_, index=NET_FEATURES).sort_values().plot.barh(ax=axes[0], color="#2f6db5")
axes[0].set_title("Feature importance — Random Forest")
ax = axes[1]; tt = nf_te.index * INTERVAL_STATISTIK
ax.plot(tt, nf_te.pkt_rate, color="#2f6db5", label="paket/s ke server")
ax.scatter(tt[preds["Random Forest"] == 1], nf_te.pkt_rate[preds["Random Forest"] == 1], color="red", s=16, zorder=3, label="RF: DDoS")
ax.scatter(tt[preds["Z-score/Entropi"] == 1], np.zeros((preds["Z-score/Entropi"] == 1).sum()), marker="|", s=80, color="black", label="z-score: alarm")
shade(ax, runs["uji"]); ax.legend(fontsize=7, ncol=2); ax.set_xlabel("waktu (s)"); ax.set_title("Prediksi pada run uji")
plt.tight_layout(); plt.savefig(OUT / "4_importance_prediksi.png", dpi=150); plt.show()

In [ ]:
#@title 🧭 Jenis serangan & port sumber
rf_type = RandomForestClassifier(n_estimators=N_POHON_RANDOM_FOREST, random_state=SEED, n_jobs=-1).fit(nf_tr[NET_FEATURES], nf_tr.phase)
p_type = rf_type.predict(nf_te[NET_FEATURES])
labels = [p for p in ["normal", "flash"] + ATTACK_TYPES if p in set(nf_tr.phase)]
src_model = RandomForestClassifier(n_estimators=N_POHON_RANDOM_FOREST, random_state=SEED, n_jobs=-1, class_weight="balanced").fit(sf_tr[SRC_FEATURES], sf_tr.label)
p_src = src_model.predict(sf_te[SRC_FEATURES])
print(f"Akurasi jenis trafik ({'/'.join(labels)}): {accuracy_score(nf_te.phase, p_type):.3f}")
print("Port sumber serangan:", {k: round(v, 3) for k, v in scores(sf_te.label, p_src).items()})
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(confusion_matrix(nf_te.phase, p_type, labels=labels), display_labels=labels).plot(ax=axes[0], colorbar=False, cmap="Purples")
axes[0].set_title("Jenis trafik (level jaringan)")
ConfusionMatrixDisplay(confusion_matrix(sf_te.label, p_src, labels=[0, 1]), display_labels=["port normal", "port penyerang"]).plot(ax=axes[1], colorbar=False, cmap="Oranges")
axes[1].set_title("Port sumber (level edge switch)")
plt.tight_layout(); plt.savefig(OUT / "4_jenis_dan_sumber.png", dpi=150); plt.show()

## 5. Demo online: deteksi + mitigasi otomatis
Serangan baru dijalankan setelah `DETIK_SEBELUM_SERANGAN`. Tiap interval: Random Forest level jaringan memutuskan ada serangan → model port sumber menandai port edge → port yang tertandai `JENDELA_BERTURUT` kali berturut-turut diberi **drop rule** (seperti `blocklist.json` pada versi Ryu).

In [ ]:
#@title ⚙️ Parameter demo
JENIS_SERANGAN_DEMO = "udp"  #@param ["syn", "udp", "icmp"]
DETIK_SEBELUM_SERANGAN = 12  #@param {type:"slider", min:4, max:60, step:2}
DURASI_SERANGAN_DEMO = 50  #@param {type:"slider", min:10, max:200, step:2}
DETIK_SESUDAH_SERANGAN = 12  #@param {type:"slider", min:4, max:60, step:2}
JENDELA_BERTURUT = 2  #@param {type:"slider", min:1, max:6, step:1}
AKTIFKAN_MITIGASI = True  #@param {type:"boolean"}

In [ ]:
#@title ▶️ Jalankan demo
if JENIS_SERANGAN_DEMO not in ATTACK_TYPES:
    print(f"⚠ '{JENIS_SERANGAN_DEMO}' tidak ikut data latih; model tetap dicoba (uji generalisasi).")
    ATTACK_TYPES_DEMO = ATTACK_TYPES + [JENIS_SERANGAN_DEMO]
else:
    ATTACK_TYPES_DEMO = ATTACK_TYPES

def online_demo():
    global ATTACK_TYPES
    saved, ATTACK_TYPES = ATTACK_TYPES, ATTACK_TYPES_DEMO
    sim = SDNSim(SEED + 99)
    for _ in range(3):
        warm = sim.step("normal")[0]
    t0, prev, pk, imp, tl, events = sim.t, warm, [], [], [], []   # prev = sampel terakhir pemanasan
    flag, blocked = defaultdict(int), {}
    total = DETIK_SEBELUM_SERANGAN + DURASI_SERANGAN_DEMO + DETIK_SESUDAH_SERANGAN
    for k in range(int(total / INTERVAL_STATISTIK)):
        el = (k + 0.5) * INTERVAL_STATISTIK
        phase = JENIS_SERANGAN_DEMO if DETIK_SEBELUM_SERANGAN <= el < DETIK_SEBELUM_SERANGAN + DURASI_SERANGAN_DEMO else "normal"
        f, p, i, util = sim.step(phase)
        pk += p; imp += i
        fd = flow_deltas(pd.DataFrame(prev + f, columns=FLOW_COLS), t0); prev = f
        w = int(fd.win.max())
        nf = network_features(fd[fd.win == w], pd.DataFrame(pk, columns=PKTIN_COLS), t0, windows=[w])
        attack = int(rf.predict(nf[NET_FEATURES])[0]) if w >= 2 else 0
        flagged = set()
        if attack:
            sf = source_features(fd[fd.win == w])
            sf = sf[[(d, pp) not in blocked for d, pp in zip(sf.dpid, sf.in_port)]]
            if len(sf):
                flagged = {(d, pp) for d, pp, y in zip(sf.dpid, sf.in_port, src_model.predict(sf[SRC_FEATURES])) if y}
        for port in list(flag):
            if port not in flagged:
                flag[port] = 0
        for port in flagged:
            flag[port] += 1
            if flag[port] >= JENDELA_BERTURUT and port not in blocked and AKTIFKAN_MITIGASI:
                blocked[port] = sim.t; sim.blocked.add(port)
                events.append(dict(t=sim.t - t0, dpid=port[0], port=port[1], host=PORT2HOST[port]))
        tl.append(dict(t=sim.t - t0, phase=phase, attack_pred=attack, util_server=util, terblokir=len(blocked),
                       **nf.iloc[0][["pkt_rate", "new_flow_rate", "src_entropy", "pktin_rate"]].to_dict()))
    ATTACK_TYPES = saved
    return pd.DataFrame(tl), pd.DataFrame(events, columns=["t", "dpid", "port", "host"]), pd.DataFrame(imp, columns=["ts", "metric", "value"]).assign(t=lambda d: d.ts - t0)

tl, ev, imp = online_demo()
a0, a1 = DETIK_SEBELUM_SERANGAN, DETIK_SEBELUM_SERANGAN + DURASI_SERANGAN_DEMO
det = tl[(tl.attack_pred == 1) & (tl.t >= a0)]
t_mit = ev.t.max() if len(ev) else a1
for e in ev.itertuples():
    print(f"  ⛔ t={e.t:5.1f}s blokir {e.host} (dpid {e.dpid}, port {e.port}) {'✔ penyerang' if e.host in ATTACKERS else '✘ client normal!'}")
print(f"Jeda deteksi            : {det.t.iloc[0] - a0:.1f} s" if len(det) else "Serangan tidak terdeteksi")
print(f"Penyerang terblokir     : {len(set(ev.host) & set(ATTACKERS))}/{len(ATTACKERS)}")
print(f"Jendela false alarm     : {int(((tl.attack_pred == 1) & ((tl.t < a0) | (tl.t > a1 + 2 * INTERVAL_STATISTIK))).sum())}")
periods = {"sebelum serangan": (0, a0), "serangan (sebelum mitigasi)": (a0, t_mit), "setelah mitigasi": (t_mit + INTERVAL_STATISTIK, a1)}
dampak = pd.DataFrame({k: {"RTT ping (ms)": imp[(imp.metric == "ping_ms") & imp.t.between(lo, hi)].value.mean(),
                           "waktu HTTP (s)": imp[(imp.metric == "curl_s") & imp.t.between(lo, hi)].value.mean(),
                           "utilisasi link server": tl[tl.t.between(lo, hi)].util_server.mean()} for k, (lo, hi) in periods.items()}).T
display(dampak.round(3))
tl.to_csv(OUT / "5_demo_timeline.csv", index=False)

fig = plt.figure(figsize=(15, 7))
gs = fig.add_gridspec(2, 3)
ax = fig.add_subplot(gs[0, :2])
ax.plot(tl.t, tl.pkt_rate, color="#2f6db5", label="paket/s ke server")
ax.scatter(tl.t[tl.attack_pred == 1], tl.pkt_rate[tl.attack_pred == 1], color="red", s=18, zorder=3, label="terdeteksi DDoS")
ax.set_ylabel("paket/s"); ax.legend(fontsize=8)
ax2 = fig.add_subplot(gs[1, :2], sharex=ax)
x = imp[imp.metric == "ping_ms"]; ax2.plot(x.t, x.value, ".", ms=4, color="#1baf7a"); ax2.set_ylabel("RTT ping (ms)", color="#1baf7a")
x = imp[imp.metric == "curl_s"]; ax3 = ax2.twinx(); ax3.plot(x.t, x.value, ".", ms=4, color="#e8743b", alpha=0.6); ax3.set_ylabel("waktu HTTP (s)", color="#e8743b")
for a in (ax, ax2):
    a.axvspan(a0, a1, color=PHASE_COLOR[JENIS_SERANGAN_DEMO], alpha=0.12)
    for t_ in ev.t:
        a.axvline(t_, color="black", ls="--", lw=1)
ax2.set_xlabel("waktu (s) — garis putus-putus = port diblokir")
axg = fig.add_subplot(gs[:, 2])
blocked_hosts = set(ev.host)
nx.draw_networkx_edges(G, POS, ax=axg, edge_color="#c9d1d9", width=1.5)
nx.draw_networkx_edges(G, POS, ax=axg, edgelist=[(h, HOST_EDGE[h]) for h in blocked_hosts], edge_color="black", width=3, style="dashed")
nx.draw_networkx_nodes(G, POS, ax=axg, node_color=node_colors(), node_size=380)
nx.draw_networkx_labels(G, POS, ax=axg, font_size=7, font_color="white")
axg.set_title("Port yang diblokir (garis putus-putus)", fontsize=10); axg.axis("off")
fig.suptitle(f"Demo online deteksi + mitigasi {JENIS_SERANGAN_DEMO.upper()} flood")
plt.tight_layout(); plt.savefig(OUT / "5_demo_online.png", dpi=150); plt.show()

## 6. Unduh hasil

In [ ]:
#@title 💾 Simpan & unduh
UNDUH_ZIP = False  #@param {type:"boolean"}
for f in sorted(OUT.iterdir()):
    print(f"  {f}  ({f.stat().st_size / 1024:.1f} KB)")
if UNDUH_ZIP:
    zip_path = shutil.make_archive("hasil_networkx_02", "zip", OUT)
    if IN_COLAB:
        from google.colab import files
        files.download(zip_path)
    else:
        print("ZIP:", zip_path)

### Ide eksperimen
1. `IP_SUMBER_PALSU = False` → flood dari IP asli: entropi IP sumber justru turun & flow baru sedikit; lihat perubahan feature importance.
2. Turunkan `LAJU_SERANGAN_PPS` ke 20–40 (serangan *low-rate*) dan bandingkan recall z-score vs ML.
3. Matikan salah satu jenis serangan saat melatih, lalu pakai jenis itu di demo → uji generalisasi model.
4. Turunkan `KAPASITAS_CONTROLLER_PKTIN` untuk melihat controller SDN sendiri menjadi bottleneck (waktu HTTP naik).
5. `AKTIFKAN_MITIGASI = False` untuk membandingkan dampak tanpa pemblokiran.